# Credit Card Fraud Prediction using Random Forest & SMOTE

In [ ]:
import urllib.request
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE

# Download Data
DATA_FILE = 'creditcard.csv'
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'

if not os.path.exists(DATA_FILE):
    print(f'Downloading data from {DATA_URL}...')
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
else:
    print(f'{DATA_FILE} already exists. Skipping download.')

df = pd.read_csv(DATA_FILE)
df.head()

In [ ]:
# Preprocess and Train
df = df.dropna()
X = df.drop('Class', axis=1)
y = df['Class']

print(f'Original dataset shape: {y.value_counts().to_dict()}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Applying SMOTE...')
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'Resampled dataset shape: {pd.Series(y_train_res).value_counts().to_dict()}')

print('Training Random Forest...')
model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train_res, y_train_res)

predictions = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print('\nClassification Report:')
print(classification_report(y_test, predictions))

# Confusion Matrix
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()